<a href="https://colab.research.google.com/github/varun1271/CAS0713-COMPUTER-NETWORKS/blob/main/CO4_AT1_CN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
n = int(input("Enter number of hosts: "))

print("\nSTAR TOPOLOGY")
print("Central Switch: SWITCH")

for i in range(1, n + 1):
    host = "Host" + str(i)
    length = int(input("Enter cable length for " + host + " (m): "))

    print(host, "----", length, "m Cat6 UTP ---- SWITCH")

Enter number of hosts: 4

STAR TOPOLOGY
Central Switch: SWITCH
Enter cable length for Host1 (m): 10
Host1 ---- 10 m Cat6 UTP ---- SWITCH
Enter cable length for Host2 (m): 20
Host2 ---- 20 m Cat6 UTP ---- SWITCH
Enter cable length for Host3 (m): 30
Host3 ---- 30 m Cat6 UTP ---- SWITCH
Enter cable length for Host4 (m): 40
Host4 ---- 40 m Cat6 UTP ---- SWITCH


In [2]:
def validate_segment(length_m):
    if length_m <= 100:
        return True
    else:
        return False
test1 = 50
test2 = 100
test3 = 120

print(test1, "m ->", validate_segment(test1))

print(test2, "m ->", validate_segment(test2))

print(test3, "m ->", validate_segment(test3))

if test3 > 100:
    print("Error: UTP segment cannot exceed 100 m")

50 m -> True
100 m -> True
120 m -> False
Error: UTP segment cannot exceed 100 m


In [3]:
import random

hosts = ["Host1", "Host2", "Host3", "Host4"]

log = open("csma_log.txt", "w")

for i in range(5):

    host1 = random.choice(hosts)
    host2 = random.choice(hosts)

    if host1 == host2:

        message = host1 + " transmitted successfully"

        print(message)
        log.write(message + "\n")

    else:

        backoff = random.randint(1, 5)

        message1 = "Collision detected: " + host1 + " and " + host2
        message2 = "Back-off interval: " + str(backoff)

        print(message1)
        print(message2)

        log.write(message1 + "\n")
        log.write(message2 + "\n")

log.close()

print("\nCSMA/CD simulation completed.")
print("Log saved as csma_log.txt")

Collision detected: Host2 and Host4
Back-off interval: 3
Collision detected: Host4 and Host3
Back-off interval: 1
Host1 transmitted successfully
Collision detected: Host4 and Host1
Back-off interval: 5
Collision detected: Host3 and Host1
Back-off interval: 4

CSMA/CD simulation completed.
Log saved as csma_log.txt


In [5]:
import socket
import threading
import time

HOST = "127.0.0.1"
PORT = 5000

mac_table = {
    "Host1": "AA:BB:CC:DD:01",
    "Host2": "AA:BB:CC:DD:02"
}


def server():

    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    server_socket.bind((HOST, PORT))
    server_socket.listen(1)

    print("Switch is waiting for Host1...")

    connection, address = server_socket.accept()

    message = connection.recv(1024).decode()

    print("\nSwitch received:", message)

    print("Switch forwards frame to Host2")

    print("\nSwitch MAC Address Table")

    for host, mac in mac_table.items():
        print(host, "->", mac)

    connection.send("Hello from Switch to Host2".encode())

    connection.close()
    server_socket.close()


def client():

    client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    client_socket.connect((HOST, PORT))

    message = "Hello from Host1"

    print("Host1 sends:", message)

    client_socket.send(message.encode())

    reply = client_socket.recv(1024).decode()

    print("Host2 receives:", reply)

    client_socket.close()
server_thread = threading.Thread(target=server)
server_thread.start()
time.sleep(1)
client()

Switch is waiting for Host1...
Host1 sends: Hello from Host1

Switch received: Hello from Host1
Switch forwards frame to Host2

Switch MAC Address Table
Host1 -> AA:BB:CC:DD:01
Host2 -> AA:BB:CC:DD:02
Host2 receives: Hello from Switch to Host2


In [6]:
class StarSwitch:

    def __init__(self):
        self.ports = []

    def add_port(self, host_id):
        self.ports.append(host_id)
        print(host_id, "connected")

    def remove_port(self, host_id):
        if host_id in self.ports:
            self.ports.remove(host_id)
            print(host_id, "removed")

    def broadcast(self, frame):

        source = frame["source"]
        message = frame["message"]

        print("\nBroadcasting:", message)

        for host in self.ports:

            if host != source:
                print("Frame delivered to", host)
switch = StarSwitch()

switch.add_port("Host1")
switch.add_port("Host2")
switch.add_port("Host3")
switch.add_port("Host4")

frame = {
    "source": "Host1",
    "message": "Hello Network"
}

switch.broadcast(frame)

switch.remove_port("Host4")

Host1 connected
Host2 connected
Host3 connected
Host4 connected

Broadcasting: Hello Network
Frame delivered to Host2
Frame delivered to Host3
Frame delivered to Host4
Host4 removed
